# **Proyecto ML - parte 1 — Envío 3**

**Grupo 18**

Integrantes:
- Hugo Lenin Guzman Monsalve
- Jose Luis Parra Fernandez
- Julian Camilo Rivera Guzman
- Sergio Alejandro Torres Melendez

## Qué cambia en este envío

El envío 2 alcanzó un accuracy de 89,15 % en validación cruzada con **Regresión Logística**
sobre una representación de TF-IDF de palabras con marcas de posición por cláusula más un
bloque de TF-IDF de caracteres de la última cláusula.

En este envío se cambia **únicamente el clasificador**: la Regresión Logística se reemplaza por
un **LinearSVC** (máquina de vectores de soporte lineal) con `C=0.5`. Todo el procesamiento de
texto y toda la representación se dejan **exactamente iguales**, a propósito: así la diferencia
que se mida es atribuible al clasificador y no a un cambio de features. La evaluación usa la
misma validación cruzada (`StratifiedKFold`, 5 particiones, `shuffle=True`, `random_state=42`),
de modo que las dos cifras son comparables partición por partición.

Las secciones 1 y 2 que vienen a continuación son las mismas del envío 2 y se repiten para que
el notebook corra de principio a fin por sí solo.

En primer lugar, se realizan las importaciones de las librerias necesarias para realizar el ciclo de machine learning

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
from functools import lru_cache

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression   # referencia del envío 2
from sklearn.svm import LinearSVC                     # modelo de este envío
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from scipy.stats import binomtest   # para comparar los dos modelos sobre las mismas reseñas

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

import matplotlib.pyplot as plt

# Descarga las stop words de NLTK (solo se necesita la primera vez en un entorno nuevo)
nltk.download('stopwords', quiet=True)

True

Luego, se leen los datos del archivo .csv que contienen las reseñas en texto crudo a las cuales se les quiere clasificar en diferentes clases.

In [3]:
df_train = pd.read_csv("train.csv")
y_train = df_train['label']

## 1. Procesamiento de texto

Teniendo en cuenta que los textos crudos no son datos estructurados, se realiza NLP (natural language processing) para poder transformarlos en vectores y, de esta manera, permitir que los algoritmos de aprendizaje puedan usarlos.

Para esto, se utilizan representaciones dispersas (matrices dispersas) que permiten representar cada frase como un vector en un espacio vectorial semantico de dimensiones $\mathbb{R}^{|V|}$, donde |V| es el tamaño del vocabulario. Se hace uso del pipeline general definido para realizar este tipo de procesamiento:
```text
Texto crudo
    ↓  Limpieza (minúsculas, tildes, puntuación)
Texto normalizado
    ↓  Tokenización
Lista de tokens
    ↓  Stop words + Stemming
Tokens procesados
    ↓  Vectorización (BoW / TF-IDF)
Vector numérico ∈ ℝ^|V|
```

Los datos tienen tildes inconsistentes ("también" y "tambien" aparecen como si fueran palabras distintas), así que se quitan antes de procesar. Las stop words pasan por la misma función para que coincidan con el texto, y se conservan las negaciones y los intensificadores porque cambian el sentido de la opinión ("no", "nunca", "sin", "muy", "poco"...).

In [4]:
stop_words_es = set(stopwords.words('spanish'))

# Palabras relevantes que no queremos eliminar porque cambiarian la interpretación del modelo
palabras_relevantes = {"no", "nunca", "jamás", "sin", "ni", "tampoco",
"pero", "aunque","muy", "bastante", "demasiado", "más", "menos", "poco"}

stemmer = SnowballStemmer('spanish')

def quitar_tildes(texto):
    # Separa cada letra de su tilde y descarta la tilde (también convierte la ñ en n)
    descompuesto = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in descompuesto if unicodedata.category(c) != 'Mn')

# Stop words sin tildes para que coincidan con el texto normalizado.
# Se restan las palabras relevantes (también sin tilde) para no perder "mas", "jamas" ni las negaciones
relevantes_sin_tilde = {quitar_tildes(p) for p in palabras_relevantes}
stop_words_sin_tilde = {quitar_tildes(p) for p in stop_words_es} - relevantes_sin_tilde

@lru_cache(maxsize=None)
def raiz(palabra):
    # Stemming con caché, porque las mismas palabras se repiten miles de veces
    return stemmer.stem(palabra)

def preprocesar_clausula(clausula):
    # A. Normalización: quitar signos de puntuación (deja solo letras y espacios)
    clausula = re.sub(r'[^\w\s]', '', clausula)

    # B. Tokenización: separar por espacios
    tokens = clausula.split()

    # C. Stopwords y Stemming: filtrar palabras vacías y extraer la raíz
    return [raiz(p) for p in tokens if p not in stop_words_sin_tilde]

Antes de aplicar el anterior proceso al texto crudo, se realiza una serie de pasos adicionales para conservar información sobre la posición de cada cláusula dentro del texto. Muchas reseñas mezclan elogios y quejas, y el sentimiento global lo define sobre todo la parte final, así que el mismo término pesa distinto según dónde aparezca ("mal" al final de la reseña no significa lo mismo que "mal" al comienzo).

Para esto, el texto se separa en cláusulas. Se corta al final de cada oración, en los punto y coma y en las comas que preceden a un conector contrastivo (sin embargo, aunque, pero, eso sí, aun así, y no). Se prefiere la cláusula a la oración porque una misma oración puede unir dos opiniones opuestas con un conector ("la batería es excelente, pero la cámara decepcionó"), y cada una debe tener su propia posición.

La última cláusula se marca con L (Last), la penúltima con P (Previous) y las demás con E (Earlier). Después, cada cláusula se preprocesa y sus tokens reciben el prefijo correspondiente. El pipeline completo para el procesamiento de los datos queda de la siguiente manera:

```text
Texto original
    ↓
Minúsculas y sin tildes
    ↓
Separación en cláusulas
    ↓
Identificación de posición
    ├── E → Cláusulas anteriores
    ├── P → Penúltima cláusula
    └── L → Última cláusula
    ↓
Preprocesamiento y tokenización
    ↓
Tokens con posición
    ↓
E_token1 E_token2 ... P_token1 P_token2 ... L_token1 L_token2 ...
```

In [5]:
# Conectores contrastivos (ya sin tildes) que abren una nueva cláusula cuando van después de una coma
CONECTORES = r'(?:sin embargo|aunque|pero|eso si|aun asi|y no)'
PATRON_CORTE = re.compile(r'(?<=[.!?])\s+|\s*;\s*|\s*,\s*(?=' + CONECTORES + r'\b)')

def separar_clausulas(texto):
    # Minúsculas, sin tildes, y corte en oraciones, punto y coma y comas antes de conectores
    texto = quitar_tildes(texto.lower().strip())
    return [c.strip() for c in PATRON_CORTE.split(texto) if c and c.strip()]

def marcar_clausulas(texto):
    # Separa en cláusulas y marca cada token según su posición
    clausulas = separar_clausulas(texto)
    n = len(clausulas)
    tokens = []
    for i, clausula in enumerate(clausulas):
        if i == n - 1:
            prefijo = "L"   # última cláusula
        elif i == n - 2:
            prefijo = "P"   # penúltima
        else:
            prefijo = "E"   # el resto
        tokens += [f"{prefijo}_{tok}" for tok in preprocesar_clausula(clausula)]
    return " ".join(tokens)

def ultima_clausula_limpia(texto):
    # Última cláusula en minúsculas, sin tildes ni puntuación y sin stemming (se usa para los caracteres)
    clausulas = separar_clausulas(texto)
    if not clausulas:
        return ""
    return re.sub(r'[^\w\s]', '', clausulas[-1])

# Versiones por lotes, que son las que recibe FunctionTransformer dentro del Pipeline.
# Se definen como funciones normales (no lambdas) para poder guardar el modelo más adelante
def marcar_clausulas_lote(textos):
    return [marcar_clausulas(t) for t in textos]

def ultima_clausula_lote(textos):
    return [ultima_clausula_limpia(t) for t in textos]

Para ver cómo funciona, se aplica el procesamiento a una reseña que tenga un conector contrastivo después de una coma.

In [6]:
mascara = df_train['text'].str.contains(r',\s*(?:pero|aunque|sin embargo)', case=False, regex=True)
ejemplo = df_train.loc[mascara, 'text'].iloc[0] if mascara.any() else df_train['text'].iloc[0]

print("Texto original:\n", ejemplo)
print("\nCláusulas:")
for c in separar_clausulas(ejemplo):
    print(" -", c)
print("\nTokens con posición:\n", marcar_clausulas(ejemplo))
print("\nÚltima cláusula (para los n-gramas de caracteres):\n", ultima_clausula_limpia(ejemplo))

Texto original:
 Compré este kit el mes pasado porque necesitaba algo para las videollamadas del trabajo, llegó en tres días a casa. La verdad, el cargador me pareció terrible para mi gusto. Lo uso casi todos los días en la oficina y ya le agarré la práctica, aunque mis compañeros se ríen de lo desordenado que tengo el escritorio. El micrófono me pareció decente para ser honesto.

Cláusulas:
 - compre este kit el mes pasado porque necesitaba algo para las videollamadas del trabajo, llego en tres dias a casa.
 - la verdad, el cargador me parecio terrible para mi gusto.
 - lo uso casi todos los dias en la oficina y ya le agarre la practica
 - aunque mis companeros se rien de lo desordenado que tengo el escritorio.
 - el microfono me parecio decente para ser honesto.

Tokens con posición:
 E_compr E_kit E_mes E_pas E_necesit E_videollam E_trabaj E_lleg E_tres E_dias E_cas E_verd E_cargador E_pareci E_terribl E_gust E_uso E_casi E_dias E_oficin E_agarr E_practic P_aunqu P_companer P_rien P

## 2. Representación del texto

Luego de aplicar todas las transformaciones necesarias al texto crudo mediante NLP, se procede a representar el texto resultante en diferentes vectores, donde cada fila representa un texto y cada columna representa una palabra o combinación de palabras (n-grama) del vocabulario. Para esto, se utiliza TF-IDF, que asigna un peso a cada término según su frecuencia dentro de la reseña y qué tan poco común es en el conjunto de reseñas (los términos que aparecen en casi todas las reseñas reciben menos peso). Ademas, se utiliza un ajuste logarítmico de la frecuencia (sublinear_tf) para reducir el impacto de palabras que aparecen muchas veces.

La representación final combina dos bloques que se concatenan con `FeatureUnion`.

- **Bloque de palabras.** TF-IDF de unigramas y bigramas sobre los tokens con posición. Se usa `lowercase=False` para que los prefijos queden como `E_`, `P_` y `L_` (por defecto el vectorizador los pasaría a minúscula).
- **Bloque de caracteres.** TF-IDF de n-gramas de caracteres (`char_wb`, de 2 a 5) calculado solo sobre la última cláusula, que es la que más pesa en el sentimiento global. Los datos tienen errores de ortografía ("encmia", "peero", "taalle") que el stemming no corrige, y los n-gramas de caracteres siguen compartiendo fragmentos con la palabra bien escrita.

Todo se arma dentro de un `Pipeline`, junto con las funciones de preprocesamiento (`FunctionTransformer`) y el clasificador. De esta forma, durante la validación cruzada el vocabulario y los pesos IDF se ajustan únicamente con los datos de entrenamiento de cada fold, y el modelo final es un solo objeto que recibe texto crudo.

In [7]:
# Bloque de palabras: marcas de posición por cláusula + TF-IDF de unigramas y bigramas
bloque_palabras = Pipeline([
    ('marcar', FunctionTransformer(marcar_clausulas_lote)),
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,      # suaviza palabras repetidas: usa 1 + log(tf)
        min_df=2,
        token_pattern=r'\S+',   # cada token separado por espacio cuenta entero, con su prefijo
        lowercase=False         # conserva los prefijos E_, P_, L_ en mayúscula
    ))
])

# Bloque de caracteres: n-gramas de caracteres de la última cláusula, robustos a errores de ortografía
bloque_caracteres = Pipeline([
    ('ultima', FunctionTransformer(ultima_clausula_lote)),
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',     # n-gramas dentro de los límites de cada palabra
        ngram_range=(2, 5),
        min_df=3,
        sublinear_tf=True
    ))
])

# Los dos bloques se concatenan en una sola matriz dispersa
representacion = FeatureUnion([
    ('palabras', bloque_palabras),
    ('caracteres', bloque_caracteres)
])

## 3. Modelo 2: LinearSVC (máquina de vectores de soporte lineal)

Un **LinearSVC** también es un clasificador lineal, pero no se entrena con el mismo criterio
que la Regresión Logística. En lugar de maximizar la verosimilitud de todas las reseñas, busca el
**hiperplano de máximo margen**: la frontera que deja la mayor distancia posible hacia las reseñas
más cercanas de cada clase. Usa pérdida *hinge*, que vale cero para las reseñas que ya están del
lado correcto y suficientemente lejos de la frontera.

Esa diferencia importa en este problema. Por el análisis del envío 2 se sabe que las reseñas
**neutrales se clasifican casi perfecto** (F1 de 98,4 %) porque solo hablan de aspectos logísticos,
mientras que casi todo el error está en la confusión **positivo ↔ negativo** de las reseñas que
mezclan elogios y quejas. Como la pérdida *hinge* ignora las reseñas que ya están bien resueltas,
el LinearSVC ajusta sus pesos usando sobre todo las reseñas **ambiguas**, que son justamente las que
hay que mejorar. La Regresión Logística, en cambio, sigue recibiendo gradiente de las reseñas
fáciles y reparte capacidad en ellas.

Se eligió porque:
- Los SVM lineales son el estándar para clasificación de texto con TF-IDF disperso y de alta
  dimensión: aquí la matriz tiene 43.138 columnas (33.160 de palabras y 9.978 de caracteres) para
  12.000 reseñas, es decir muchas más features que ejemplos.
- La solución depende solo de los vectores de soporte, así que es menos sensible a las reseñas
  fáciles y redundantes.
- Es igual de rápido de entrenar que la Regresión Logística y mantiene la interpretabilidad de los
  pesos por término.

Sobre el valor de `C=0.5`: `C` es el inverso de la regularización, igual que en la Regresión
Logística. El envío 2 usaba `C=10` (regularización débil), pero con 43.138 features y 12.000
reseñas conviene regularizar más fuerte, para que un n-grama que aparece en muy pocas reseñas no
termine con un peso grande y decida la clase por sí solo. `C=0.5` penaliza más los pesos y obliga
al modelo a apoyarse en términos que se repiten. Los valores de `C` **no se exploran en este envío**:
la búsqueda de hiperparámetros es el objetivo del envío 4, y hacerla aquí impediría atribuir la
diferencia al cambio de clasificador.

Una limitación que conviene dejar anotada: el LinearSVC **no entrega probabilidades**
(no tiene `predict_proba`, solo `decision_function`). Para este envío no afecta, porque la métrica es
accuracy, pero hay que tenerlo en cuenta si en el envío 5 se arma un ensamble por promedio de
probabilidades.

In [ ]:
# Misma validación cruzada del envío 2: 5 particiones estratificadas, mezcladas con semilla fija.
# Al reutilizar el mismo objeto cv, los dos modelos se evalúan sobre EXACTAMENTE las mismas particiones
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modelo del envío 2, que se conserva solo como referencia para comparar
modelo_logreg = Pipeline([
    ('rep', representacion),
    ('clf', LogisticRegression(C=10, max_iter=2000, random_state=42))
])

# Modelo de este envío: misma representación, clasificador distinto
modelo_svc = Pipeline([
    ('rep', representacion),
    ('clf', LinearSVC(
        C=0.5,           # Inverso de la regularización: más bajo que el C=10 del envío 2, penaliza más los pesos
        random_state=42  # Semilla para que los resultados sean replicables
    ))
])

scores_logreg = cross_val_score(modelo_logreg, df_train['text'], y_train, cv=cv, scoring='accuracy')
scores_svc = cross_val_score(modelo_svc, df_train['text'], y_train, cv=cv, scoring='accuracy')

print("Regresión Logística C=10  -> por fold:", scores_logreg.round(4),
      f"| promedio: {scores_logreg.mean():.4f} (+/- {scores_logreg.std():.4f})")
print("LinearSVC C=0.5           -> por fold:", scores_svc.round(4),
      f"| promedio: {scores_svc.mean():.4f} (+/- {scores_svc.std():.4f})")
print(f"\nDiferencia promedio: {(scores_svc.mean() - scores_logreg.mean()) * 100:+.2f} puntos porcentuales")
print(f"Folds en los que gana LinearSVC: {(scores_svc > scores_logreg).sum()} de {len(scores_svc)}")

### Análisis de resultados por clase

Igual que en el envío 2, el accuracy promedio no dice en qué clases se equivoca el modelo. Con
`cross_val_predict` se obtiene, para cada una de las 12.000 reseñas, la predicción del modelo del
fold en el que esa reseña quedó como validación, así que ninguna predicción viene de un modelo que
ya había visto esa reseña.

In [ ]:
# Predicción de cada reseña con el modelo del fold donde quedó como validación
y_pred_svc = cross_val_predict(modelo_svc, df_train['text'], y_train, cv=cv)

# Precisión, recall y F1 por clase
print(classification_report(y_train, y_pred_svc, digits=4))

# Matriz de confusión: filas = clase real, columnas = clase predicha
ConfusionMatrixDisplay.from_predictions(y_train, y_pred_svc, cmap='Blues')
plt.title("Matriz de confusión - LinearSVC C=0.5 (CV 5 folds)")
plt.show()

# Cuántos errores son confusiones entre positivo y negativo
errores_svc = (y_train != y_pred_svc)
opinion = y_train.isin(["positivo", "negativo"]) & pd.Series(y_pred_svc, index=y_train.index).isin(["positivo", "negativo"])
print(f"Errores totales: {errores_svc.sum()} | confusiones positivo<->negativo: {(errores_svc & opinion).sum()}")

### Comparación contra el envío 2 sobre las mismas reseñas

Comparar los dos promedios no basta: una diferencia de pocas décimas puede ser ruido de la
partición. Como los dos modelos se evaluaron con los mismos folds, se pueden comparar **reseña por
reseña** y contar en cuántas acierta uno y falla el otro. Ese es el conteo que usa la prueba de
**McNemar**; las reseñas en las que ambos aciertan o ambos fallan no aportan información sobre
cuál es mejor y se descartan.

La prueba se hace en su versión exacta (binomial): si los dos modelos fueran igual de buenos, de
los desacuerdos la mitad debería caer de cada lado.

In [ ]:
# Predicciones del modelo del envío 2 sobre los mismos folds
y_pred_logreg = cross_val_predict(modelo_logreg, df_train['text'], y_train, cv=cv)

acierta_logreg = (y_pred_logreg == y_train.values)
acierta_svc = (y_pred_svc == y_train.values)

solo_svc = int((~acierta_logreg & acierta_svc).sum())      # las arregla LinearSVC
solo_logreg = int((acierta_logreg & ~acierta_svc).sum())   # las rompe LinearSVC

print(f"Las dos predicen lo mismo en {(y_pred_logreg == y_pred_svc).sum()} de {len(y_train)} reseñas")
print(f"Solo acierta LinearSVC:        {solo_svc}")
print(f"Solo acierta Reg. Logística:   {solo_logreg}")
print(f"Ambos aciertan:                {(acierta_logreg & acierta_svc).sum()}")
print(f"Ambos fallan:                  {(~acierta_logreg & ~acierta_svc).sum()}")

# Prueba de McNemar exacta sobre los desacuerdos
p_valor = binomtest(solo_svc, solo_svc + solo_logreg, 0.5).pvalue
print(f"\nMcNemar exacto sobre los {solo_svc + solo_logreg} desacuerdos: p = {p_valor:.4f}")

# Tabla resumen de los dos envíos
resumen = pd.DataFrame({
    "Modelo": ["Envío 2 - Reg. Logística C=10", "Envío 3 - LinearSVC C=0.5"],
    "Accuracy CV": [scores_logreg.mean(), scores_svc.mean()],
    "Desv. entre folds": [scores_logreg.std(), scores_svc.std()],
    "Errores (de 12.000)": [int((~acierta_logreg).sum()), int((~acierta_svc).sum())],
})
print()
print(resumen.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Conclusiones del Modelo 2

El LinearSVC con `C=0.5`, sobre la misma representación del envío 2, alcanza un accuracy promedio
de **89,39 %** en validación cruzada (±0,37 % entre folds), frente al **89,15 %** de la Regresión
Logística: una mejora de **+0,24 puntos porcentuales**, y los errores bajan de 1.302 a 1.273 sobre
las 12.000 reseñas.

| Modelo | Accuracy CV | Desv. entre folds | Errores |
| --- | --- | --- | --- |
| Envío 2 — Reg. Logística C=10 | 89,15 % | ±0,34 % | 1.302 |
| Envío 3 — LinearSVC C=0,5 | **89,39 %** | ±0,37 % | **1.273** |

**La mejora es consistente pero pequeña.** El LinearSVC gana en los **5 de 5 folds**, lo cual
descarta que el resultado dependa de una partición afortunada. Sin embargo, la comparación reseña
por reseña muestra que de los 227 desacuerdos, 128 los arregla el LinearSVC y 99 los rompe: la
prueba de McNemar exacta da **p = 0,063**, por encima del umbral habitual de 0,05. La lectura
honesta es que el LinearSVC es **igual o algo mejor**, con la evidencia apuntando de forma
consistente a su favor, pero sin que la diferencia sea concluyente con 12.000 reseñas. Los dos
modelos coinciden en 11.756 de las 12.000 predicciones, es decir el 98 %: están aprendiendo
prácticamente lo mismo, como era de esperarse al compartir la representación.

Resultados por clase, comparados con el envío 2:

| Clase | F1 envío 2 | F1 envío 3 | Cambio |
| --- | --- | --- | --- |
| Neutral | 98,37 % | 98,11 % | −0,26 |
| Negativo | 85,33 % | 85,80 % | +0,47 |
| Positivo | 85,04 % | 85,41 % | +0,37 |

El cambio tiene una interpretación clara y coherente con el diagnóstico del envío 2:

- **El LinearSVC mejora donde estaba el problema.** Las confusiones positivo ↔ negativo bajan de
  1.184 a 1.136 (−48). Esas reseñas ambiguas son exactamente las que quedan cerca de la frontera,
  o sea las que la pérdida *hinge* usa para ajustar los pesos. Es el efecto que se esperaba del
  cambio de criterio de entrenamiento.
- **Y empeora un poco donde no había problema.** En neutral el recall sube (99,33 % → 99,66 %) pero
  la precisión baja (97,42 % → 96,61 %): el LinearSVC se vuelve más propenso a declarar neutral, así
  que recupera algunas neutrales más pero se lleva por delante algunas reseñas con opinión. Es un
  intercambio desfavorable en esa clase, porque neutral ya estaba casi resuelta.
- **El cuello de botella no cambió.** El 89 % de los errores restantes (1.136 de 1.273) sigue siendo
  confusión positivo ↔ negativo. Cambiar el clasificador no podía arreglar eso de fondo: el límite
  está en la **representación**, que no distingue cuál de las varias opiniones de una reseña es la
  decisiva. Mientras una queja intercalada y la valoración final se representen con el mismo tipo de
  token, ningún clasificador lineal sobre estas features va a separarlas mejor.

**Qué sigue.** El margen de este envío está agotado: la representación es la misma y ya se probaron
los dos clasificadores lineales razonables. El envío 4 busca hiperparámetros sobre este mismo
pipeline con `RandomizedSearchCV` (rangos de n-gramas, `min_df`, `max_df`, peso del bloque de
caracteres, `C` y `class_weight`), lo que debería recuperar parte de lo que se pierde en neutral y
afinar el `C` que aquí se fijó a mano. La mejora de fondo sobre la confusión positivo ↔ negativo
queda para el envío 5, donde hay que modelar explícitamente **cuál opinión manda** dentro de la
reseña.

> **Nota posterior.** El plan cambió en los envíos siguientes. El envío 4 no hizo la búsqueda de
> hiperparámetros: cambió la **representación** (roles de opinión O1/O2/O3/F), y en su sección 6
> explica por qué la búsqueda ya no tenía margen. El envío 5 agregó el tratamiento de negaciones y
> cambió el clasificador a `RidgeClassifier`. No se armó un ensamble por probabilidades.

## 4. Generación de las predicciones y archivo final

Como todo el preprocesamiento, los vectorizadores y el clasificador están dentro de un solo
`Pipeline`, basta reentrenarlo con el 100 % de train y llamar a `predict` sobre el texto crudo de
eval. El archivo se guarda con un nombre distinto al del envío 2 para poder subir los dos y
comparar en el leaderboard.

In [ ]:
# 4. Generación de Predicciones para Kaggle

# Re-entrenar el pipeline completo con el 100% de los datos de train
modelo_svc.fit(df_train['text'], y_train)

# Cargar los archivos de evaluación y formato
df_eval = pd.read_csv("eval.csv")
df_sub = pd.read_csv("sample_submission.csv")

# Generar predicciones (el pipeline aplica cláusulas, TF-IDF y clasificador sobre el texto crudo)
predicciones = modelo_svc.predict(df_eval['text'])

# Asignar los resultados a la columna 'answer'
df_sub['answer'] = predicciones

# Distribución de clases predichas
print("Distribución de predicciones (proporción):")
print(df_sub["answer"].value_counts(normalize=True).round(4))

# Cuántas predicciones cambian respecto al envío 2
sub_v2 = pd.read_csv("submission_logreg_v2_grupo18.csv")
cambios = (sub_v2['answer'].values != df_sub['answer'].values).sum()
print(f"\nFilas: {len(df_sub)} | predicciones distintas al envío 2: {cambios} ({cambios / len(df_sub):.1%})")

# Exportar
df_sub.to_csv("submission_linearsvc_v3_grupo18.csv", index=False)
print("\nArchivo generado: submission_linearsvc_v3_grupo18.csv")

El archivo de salida cambia **71 de las 3.000 predicciones** (2,4 %) respecto al envío 2, con una
distribución de clases prácticamente idéntica (35,7 % negativo, 35,3 % positivo, 29,0 % neutral).
Un cambio tan pequeño es coherente con la mejora medida en validación cruzada: no hay que esperar
un salto grande en el leaderboard.

Importante para decidir qué envío vale: la elección entre el envío 2 y el envío 3 debe hacerse con
la **validación cruzada**, no con el score público del leaderboard. El público se calcula sobre una
fracción de los datos de test y una diferencia de 0,2 puntos entra dentro de su ruido; la CV usa las
12.000 reseñas de train y, además, ya mostró que la ventaja se repite en los 5 folds.